<center> <h1>
    <b>
        Mid-term Project:
    </b><br>
        Sequential Image Stitching
</h1> </center>

## **Table of Contents**

<!-- TODO -->
toc placeholder

## **Team Information**
<b> GroupID: 03 </b>
| Student Name          | Student ID |
|-----------------------|------------|
| Nguyen Hoang Minh Tam | 23127017   |
| Nguyen Manh Cuong     | 23127034   |
| Phan Nhat Thien Nhan  | 23127096   |

<b> Contributions </b>
| Student | Tasks |
|---------|-------|
| Tam     | + standardize Codebase and Dataset<br> + write `homography.cu`, `warp.cu`<br> + 
| Cuong   | + synthesize Image<br> + write `keypoint.cu`, `matching.cu`<br> + write .ipynb Scripts and Report
| Nhan    | + organize Repository<br> + write `preprocess.cu`, `describe.cu`<br> +

---
## I. **Problem Statement**

Panorama stitching combines overlapping photos into one wide image, as phone cameras do in panorama mode. The pipeline finds distinctive points in each photo, matches them, estimates the geometric transform between the photos, and warps and blends them together.

In this project, our group attempt to address the problem: **Stiching two distinct overlapping Images** via this six-stage process:
1. preprocessing              : transform Images into an easy-to-process datas
2. keypoint detection         : detect visual-cue feature in both Images
3. feature description        : describe the neighbouring despictor around each keypoints
4. feature matching           : find the overlapping keypoints in both Images 
5. homography estimation      : estimate the projective transformation
6. image warping and blending : blend both Images into and oput Image

In each stage in the pipeline, we will clearly describe our Algorithms, The decision Process, and Trade-offs we made toward the final Ouput.

---
## II. **Methods**

### 1. **Datasets**

### 2. **Sequential Version** (VO)

----
## III. **Implmentation** (Colab)

---
## IV. **Result & Analysis**

---
## V. **Conclusion & Future Works** (on GPU)

---
## A. **Appendix**
### A.1 **Methods, Formulations, Decision Clarification & Implemenation Details**

#### 1. Phase 1: Image Preprocessing 

##### 1.1 Objective & Role in Pipeline
The raw input consists of an 8-bit interleaved RGB image $[W \times H \times 3]$. Preprocessing prepares this image for feature extraction through two core steps:
1. **Color Decoupling**: Project trichromatic RGB data into a single-channel floating-point grayscale image $[W \times H]$ ($I \in [0.0, 255.0]$).
2. **Noise Reduction**: Smooth the grayscale image using Gaussian filtering. Without smoothing, sensor noise and high-frequency textures would introduce spurious, non-repeatable keypoint responses in subsequent gradient-based detectors .


##### 1.2 Grayscale Conversion
* **Method Chosen**: ITU-R Recommendation BT.601-7 Standard Luma Projection.
* **Mathematical Formulation**:
  The RGB color tuple $(R, G, B)$ is mapped to luminance scalar $Y$ via:
  $$Y = 0.299 \cdot R + 0.587 \cdot G + 0.114 \cdot B$$
  The coefficients sum to unity ($0.299 + 0.587 + 0.114 = 1.000$), preserving the dynamic range such that $(255, 255, 255)$ maps to exactly $255.0$.

* **Why I choose this**:
  * *Parallel Efficiency*: Evaluates in a single memory pass with 3 multiply-accumulate (MAC) operations per pixel, fully coalesced across GPU warps.

* **Implementation Details**:
  * Input bytes are indexed in interleaved order: `rgbIdx = (r * width + c) * 3`.
  * Each channel is cast to `float` and accumulated into a contiguous 1D array `std::vector<float> gray(width * height)`.


##### 1.3 Gaussian Smoothing & Separable Convolution
* **Method Chosen**: 2-Pass Separable 1D Gaussian Convolution with $\sigma = 2.0$ and radius $r = 3$ (kernel diameter $K = 2r + 1 = 7$).
* **Mathematical Formulation**:
  1. **Continuous 1D Gaussian Function**:
     $$G(x; \sigma) = \frac{1}{\sqrt{2\pi}\sigma} \exp\left(-\frac{x^2}{2\sigma^2}\right)$$
  2. **Discrete Kernel Weights**: Evaluated at integer offsets $k \in \{-3, -2, -1, 0, 1, 2, 3\}$:
     $$w_k = \exp\left(-\frac{k^2}{2\sigma^2}\right)$$
  3. **DC Gain Normalization**:
     $$K_k = \frac{w_k}{\sum_{j=-r}^{r} w_j} \implies \sum_{k=-r}^{r} K_k = 1.0$$
     This guarantees that filtering a region of uniform intensity $C$ satisfies $\sum K_k \cdot C = C$, preventing artificial brightness drift.
  4. **The Separability Theorem**:
     Because the 2D Gaussian kernel factors into two 1D Gaussians ($G_{2D}(x, y) = G_{1D}(x) \cdot G_{1D}(y)$), 2D convolution factors into two successive 1D passes:
     $$I_{\text{smooth}}(x, y) = \left( (I * G_x) * G_y \right)(x, y)$$

* **Why I choose this**:
  * *vs. Direct 2D Gaussian ($K \times K$)*:
    * For $K = 7$, a 2D filter requires $K^2 = 49$ calculations/pixel.
    * A separable filter requires $2K = 14$ calculations/pixel, which is a big gap of time speed.
    * *GPU Benefit*: On CUDA, separable passes use 1D shared-memory tiles with 1D halos, drastically reducing shared memory footprint and halo divergence compared to 2D square halos.
  * *vs. Image Pyramid (Multi-Scale)*:
    * Image pyramids require building and maintaining multiple downsampled layers, adding significant memory allocations and inter-scale synchronization.

* **Boundary Handling (`clampInt`)**:
    $$x_{\text{sample}} = \operatorname{clamp}(c + k, 0, W - 1) = \min(\max(c + k, 0), W - 1)$$

* **Implementation Details**:
  * **Pass 1 (Horizontal)**: Convolves along rows into `tempBuffer[r * width + c]`, achieving contiguous linear row memory reads.
  * **Pass 2 (Vertical)**: Convolves along columns from `tempBuffer` into the output `blurred.data[r * width + c]`.

#### 3. Phase 3: Feature Description 

##### 3.1 Objective & Role in Pipeline
Stage 2 provides candidate keypoint positions $(x, y)$, but keypoint coordinates alone cannot be matched across images. Stage 3 computes a compact **256-bit binary descriptor** for each keypoint. To achieve robustness against camera tilt and rotation, the patch orientation is computed via the **Intensity Centroid**, and the descriptor pattern is steered accordingly (Rotated BRIEF / rBRIEF, the descriptor component of ORB).


##### 3.2 Deterministic Sampling Pattern via 32-bit LCG
* **Method Chosen**: 256 test coordinate pairs sampled uniformly across a $31 \times 31$ patch ($r = 15$) using a deterministic 32-bit Linear Congruential Generator with fixed seed `p.briefSeed = 2026`.
* **Mathematical Formulation**:
  $$X_{n+1} = (a \cdot X_n + c) \pmod m$$
  * Modulus: $m = 2^{32}$ (implicit via 32-bit unsigned overflow).
  * Multiplier: $a = 1664525$ (Knuth, *TAOCP Vol. 2*).
  * Increment: $c = 1013904223$ (odd constant).
  * Range mapping: $x, y = (\text{lcgNext} \bmod (2r+1)) - r \in [-15, 15]$.


##### 3.3 Keypoint Orientation via Intensity Centroid
* **Method Chosen**: Rosin (1999) / ORB (Rublee et al., 2011) Intensity Centroid over a circular patch of radius $r = 15$.
* **Mathematical Formulation**:
  1. **2D Spatial Image Moments** over circular mask $\mathcal{B}_r = \{(u, v) \in \mathbb{Z}^2 \mid u^2 + v^2 \le r^2\}$:
     $$m_{10} = \sum_{(u, v) \in \mathcal{B}_r} u \cdot I(c_x + u, c_y + v)$$
     $$m_{01} = \sum_{(u, v) \in \mathcal{B}_r} v \cdot I(c_x + u, c_y + v)$$
  2. **Centroid Vector**: The intensity center of mass is $C = \left(\frac{m_{10}}{m_{00}}, \frac{m_{01}}{m_{00}}\right)$.
  3. **Dominant Orientation**: The orientation angle $\theta$ is the direction of vector $C$:
     $$\theta = \operatorname{atan2}(m_{01}, m_{10}), \quad \theta \in [-\pi, \pi]$$

* **Why I choose this **:
  * *vs. SIFT-style Gradient Orientation Histograms*: SIFT requires computing per-pixel gradient magnitudes, gradient orientations, Gaussian weighting, and building a 36-bin histogram with peak interpolation. Intensity Centroid computes two basic accumulation sums ($m_{10}, m_{01}$) and a single `atan2`, saving massive arithmetic work and eliminating branch divergence on GPU.


##### 3.4 Rotated BRIEF (rBRIEF) Descriptor & Bitpacking
* **Method Chosen**: Steered BRIEF (Calonder et al., 2010; Rublee et al., 2011) with 256 binary tests packed into 8 `uint32_t` words.
* **Mathematical Formulation**:
  1. **2D Coordinate Steering**: To achieve rotation invariance, canonical test coordinates $(x, y)$ are rotated by orientation angle $\theta$ via the 2D rotation matrix:
     $$\begin{bmatrix} rx \\ ry \end{bmatrix} = \begin{bmatrix} \cos\theta & -\sin\theta \\ \sin\theta & \cos\theta \end{bmatrix} \begin{bmatrix} x \\ y \end{bmatrix} = \begin{bmatrix} x\cos\theta - y\sin\theta \\ x\sin\theta + y\cos\theta \end{bmatrix}$$
     Discrete sampling locations:
     $$p = \left( c_x + \operatorname{round}(rx), \; c_y + \operatorname{round}(ry) \right)$$
  2. **Binary Intensity Test**:
     $$\tau(I; p_1, p_2) = \begin{cases} 1 & \text{if } I(p_1) < I(p_2) \\ 0 & \text{otherwise} \end{cases}$$
  3. **Bitpacking**: 256 tests are packed into 8 unsigned 32-bit integers:
     $$\text{wordIdx} = \lfloor i / 32 \rfloor \in [0, 7], \quad \text{bitIdx} = i \bmod 32$$
     $$\text{desc.w}[\text{wordIdx}] \mathrel{|}= (1 \ll \text{bitIdx})$$

* **Why I choose this**:
  * *vs. Standard BRIEF*: Standard BRIEF uses unrotated axis-aligned test points and **fails completely under in-plane camera rotation**. Steered BRIEF rotates the sampling grid, maintaining matching repeatability under arbitrary tilt.
  * *vs. SIFT (128-dimensional Float Descriptors)*:
    * *Memory Footprint*: SIFT requires $128 \times 4\text{ bytes} = \mathbf{512\text{ bytes}}$ per keypoint; rBRIEF requires $256\text{ bits} = \mathbf{32\text{ bytes}}$ (**$16\times$ less memory bandwidth**).
    * *Matching Complexity (Stage 4)*: SIFT matching requires computing floating-point Euclidean distance ($L_2$ norm). rBRIEF matching uses Hamming distance via bitwise `XOR` and hardware `POPCOUNT` (`__popc` on GPU):

      Evaluating 8 x 32-bit integer operations is faster than 128 floating-point operations, and compact 32-byte descriptors fit efficiently into GPU shared memory.

* **Boundary Safety & Synchronization**:
  * If any rotated coordinate falls outside $[0, W) \times [0, H)$, the keypoint is dropped.
  * Dropping invalid keypoints synchronously from both `kps` and `desc` maintains a strict 1-to-1 invariant ($\text{kps}[i] \leftrightarrow \text{desc}[i]$) required for Stage 4 matching and Stage 5 RANSAC.
